In [0]:
# Test — Day 8-9 Governance (RLS/CLS) & Usage Analysis
# Requires 19_governance_rls.sql and 20_governance_cls.sql to have run first
# (they create the views this test queries).

import unittest

CATALOG = "vstone_catalog"
SECURITY = "security"


class GovernanceDay8Tests(unittest.TestCase):

    def test_rls_views_exist_and_have_rows(self):
        for view in ["rls_fact_street_readings", "rls_monthly_street_trend"]:
            with self.subTest(view=view):
                count = spark.table(f"{CATALOG}.{SECURITY}.{view}").count()
                self.assertGreater(count, 0, f"{view} is empty — did 19_governance_rls.sql run?")

    def test_cls_views_exist_and_have_rows(self):
        for view in ["cls_dim_street", "cls_dim_node_location", "cls_fact_citizen_reports"]:
            with self.subTest(view=view):
                count = spark.table(f"{CATALOG}.{SECURITY}.{view}").count()
                self.assertGreater(count, 0, f"{view} is empty — did 20_governance_cls.sql run?")

    def test_cls_dim_street_masks_coordinates_for_non_privileged_caller(self):
        """
        This test's own identity determines the outcome: if run as
        admin_group/safety_team, coordinates are visible (test passes
        trivially, admin sees real data). If run as a non-member, they
        must be NULL. Either way, confirms the CASE WHEN logic is wired
        correctly, not a placeholder.
        """
        df = spark.table(f"{CATALOG}.{SECURITY}.cls_dim_street")
        is_privileged = spark.sql(
            "SELECT IS_MEMBER('admin_group') OR IS_MEMBER('safety_team') AS p"
        ).collect()[0]["p"]
        null_coords = df.filter("latitude IS NULL AND longitude IS NULL").count()
        total = df.count()
        if is_privileged:
            self.assertEqual(null_coords, 0, "Privileged caller should see all real coordinates.")
        else:
            self.assertEqual(null_coords, total, "Non-privileged caller should see ALL coordinates masked.")

    def test_cls_fact_citizen_reports_redacts_or_reveals_consistently(self):
        df = spark.table(f"{CATALOG}.{SECURITY}.cls_fact_citizen_reports")
        redacted_count = df.filter("message = '**REDACTED**'").count()
        total = df.count()
        is_privileged = spark.sql(
            "SELECT IS_MEMBER('admin_group') OR IS_MEMBER('safety_team') AS p"
        ).collect()[0]["p"]
        if is_privileged:
            self.assertEqual(redacted_count, 0, "Privileged caller should see real message text.")
        else:
            self.assertEqual(redacted_count, total, "Non-privileged caller should see ALL messages redacted.")

    def test_usage_analysis_queries_run_without_error(self):
        """system.billing.usage may be empty on a brand-new Free Edition
        workspace with no billing history yet — this checks the queries
        are valid, not that they return rows."""
        try:
            spark.sql("""
                SELECT usage_date, sku_name, SUM(usage_quantity) AS total_dbus
                FROM system.billing.usage
                WHERE usage_date >= CURRENT_DATE() - INTERVAL 7 DAYS
                GROUP BY 1, 2
            """).collect()
        except Exception as e:
            self.fail(f"Usage analysis query failed: {e}")


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(GovernanceDay8Tests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise Exception("Day 8-9 governance tests FAILED — see output above.")
